In [1]:
import sys; sys.path.append("..")          
import pandas as pd                                                  
from src.data_loader import PROCESSED_DIR    
from src.preprocessing import glu_map, a1c_map, med_map, med_cols, encode_labs, encode_meds, icd9_group, group_diagnoses, onehot_cols,numeric_cols,build_preprocessor

In [2]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")      
print(X_train.shape)         

(69613, 44)


In [3]:
# columns that are not numeric
text_cols = X_train.select_dtypes(exclude="number").columns       
print(X_train[text_cols].nunique().sort_values(ascending=False))  

diag_3                      748
diag_2                      697
diag_1                      683
medical_specialty            70
payer_code                   18
race                          6
glimepiride                   4
glyburide-metformin           4
insulin                       4
rosiglitazone                 4
pioglitazone                  4
glyburide                     4
glipizide                     4
chlorpropamide                4
nateglinide                   4
repaglinide                   4
metformin                     4
A1Cresult                     4
max_glu_serum                 4
acarbose                      3
tolazamide                    3
acetohexamide                 2
tolbutamide                   2
miglitol                      2
troglitazone                  2
glipizide-metformin           2
metformin-rosiglitazone       2
metformin-pioglitazone        2
glimepiride-pioglitazone      1
dtype: int64


In [4]:
print(glu_map)
print(X_train["max_glu_serum"].value_counts())
print(encode_labs(X_train)["max_glu_serum"].value_counts())

{'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
max_glu_serum
None    66034
Norm     1803
>200      982
>300      794
Name: count, dtype: int64
max_glu_serum
0    66034
1     1803
2      982
3      794
Name: count, dtype: int64


In [5]:
print(med_map)
print(X_train["insulin"].value_counts())
print(encode_meds(X_train)["insulin"].value_counts())

{'No': 0, 'Down': 1, 'Steady': 2, 'Up': 3}
insulin
No        32507
Steady    21209
Down       8262
Up         7635
Name: count, dtype: int64
insulin
0    32507
2    21209
1     8262
3     7635
Name: count, dtype: int64


In [6]:
print(group_diagnoses(X_train)["diag_1"].value_counts())
print(group_diagnoses(X_train)["diag_2"].value_counts())
print(group_diagnoses(X_train)["diag_3"].value_counts())

diag_1
Circulatory        20671
Other              12443
Respiratory         9845
Digestive           6595
Diabetes            6092
Injury              4805
Genitourinary       3504
Musculoskeletal     3486
Neoplasms           2160
Unknown               12
Name: count, dtype: int64
diag_2
Circulatory        21776
Other              18178
Diabetes            8990
Respiratory         7275
Genitourinary       5754
Digestive           2825
Injury              1678
Neoplasms           1622
Musculoskeletal     1260
Unknown              255
Name: count, dtype: int64
diag_3
Circulatory        20753
Other              20004
Diabetes           11962
Respiratory         4916
Genitourinary       4479
Digestive           2700
Musculoskeletal     1331
Injury              1309
Neoplasms           1152
Unknown             1007
Name: count, dtype: int64


In [7]:
print(icd9_group("250.83"), icd9_group("428"), icd9_group("V57"), icd9_group("Unknown"))   

Diabetes Circulatory Other Unknown


In [8]:
pre = build_preprocessor()
X_train_enc = pre.fit_transform(X_train)
print(X_train_enc.shape)
pre

(69613, 138)


,steps,"[('prep', ...), ('encode', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('labs', ...), ('meds', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function enc...t 0x11e7ceb90>
,inverse_func,None
,validate,False


In [9]:
print("Non-numeric columns left:", X_train_enc.select_dtypes(exclude="number").shape[1])   # must be 0
print("NaNs:", X_train_enc.isna().sum().sum())                                              # must be 0
print(X_train_enc[numeric_cols].agg(["mean", "std"]).round(3))                              # 0 and 1
print(X_train_enc.filter(like="race_").head(3))

Non-numeric columns left: 0
NaNs: 0
      time_in_hospital  num_lab_procedures  num_procedures  num_medications  \
mean               0.0                 0.0             0.0              0.0   
std                1.0                 1.0             1.0              1.0   

      number_outpatient  number_emergency  number_inpatient  number_diagnoses  
mean                0.0               0.0               0.0              -0.0  
std                 1.0               1.0               1.0               1.0  
              race_AfricanAmerican  race_Asian  race_Caucasian  race_Hispanic  \
encounter_id                                                                    
149190                         0.0         0.0             1.0            0.0   
64410                          1.0         0.0             0.0            0.0   
500364                         0.0         0.0             1.0            0.0   

              race_Other  race_Unknown  
encounter_id                           